## Step 1: Import Libraries & API Keys

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr
import json
import requests

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is missing.")

/Users/daniellechoi/anaconda3/envs/ai-env313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2: Set up Pushover

In [2]:
load_dotenv()

True

In [3]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "http://api.pushover.net/1/messages.json"

In [4]:
if pushover_user is None:
    raise Exception("User is missing.")
if pushover_token is None:
    raise Exception("Pushover token is missing.")

In [5]:
# Test pushover
def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
send_notification("Hello to myself, from this AI Engineering training.")

## Step 3: Describe Pushover as an LLM tool

In [6]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the user's phone via Pushover. Use this to alert the user about important events, completed tasks, or time-sensitive information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

## Step 4: Add Pushover to the list of tools for the LLM

In [7]:
tools = [{
    "type": "function",
    "function": send_notification_function
}]

## Step 5: Calling the tool from an LLM

In [22]:
def handle_tool_call(tool_calls):
    tool_call_results = []
    content = ""

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        # print(f"Calling function: {function_name}") # for future debugging

        # Route to the appropriate function based on the function name
        if function_name == "send_notification":
            send_notification(args["message"])
            content = f"Notification sent: {args['message']}"
        # elif function_name == "insert_function_name_2":
        #     content = insert_function_name_2(args["message"])
        # elif function_name == "insert_function_name_3":
        #     content = insert_function_name_3(args["message"])
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id,
        }
        tool_call_results.append(tool_call_result)

    return tool_call_results

In [23]:
client = OpenAI()
messages = [{
        "role": "user", "content": "Please send me two notifications telling me what amazing progress \
        I am making on the AI Engineering training."
    }]

response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages=messages,
    tools = tools,
    tool_choice="auto" # none, required, {}, auto
)

message = response.choices[0].message

# Check if the model wants to call a tool
if message.tool_calls:
    tool_result = handle_tool_call(message.tool_calls) # whole list of tool calls on purpose
    messages.append(message)
    messages.extend(tool_result) # changed append to extend to add all tool call results
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages=messages,
        # tools = tools, # will add this in the future
        # tool_choice="auto",
    )
    message=response.choices[0].message

print(message.content)

I've sent you two notifications celebrating the amazing progress you're making on the AI Engineering training. Keep up the fantastic work!


In [14]:
# Done with use

# if message.tool_calls:
#     tool_call = message.tool_calls[0]
#     args = json.loads(tool_call.function.arguments)

#     # Actually send the notification
#     send_notification(args["message"])
#     print(f"Send notification: {args["message"]}")

# else:
#     print(message.content)